
# Multimodal Product Matching Baseline

Product matching using pretrained EfficientNet-B0 image embeddings and TF-IDF text features.

The pipeline combines cosine-distance retrieval, threshold filtering, and union-based multimodal fusion. Performance is evaluated using Mean F1.



## 1. Setup

Repository setup, dependencies, and Kaggle authentication.


In [1]:
!git clone https://github.com/rimfa/shopee-product-matching.git /content/shopee-product-matching

Cloning into '/content/shopee-product-matching'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 52 (delta 10), reused 34 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 1.68 MiB | 8.00 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [2]:
%cd /content/shopee-product-matching

/content/shopee-product-matching


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install kagglehub

In [5]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [6]:
import numpy as np
import pandas as pd
import os

from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

from src.embeddings import get_tfidf
from src.retrieval import image_retrieval, text_retrieval
from src.threshold_filtering import threshold_filtering
from src.fusion import multimodal_fusion
from src.validator import evaluate_predictions


## 2. Data Preparation

Load the Shopee dataset and precomputed EfficientNet-B0 embeddings. Reconstruct the train/validation split using saved `posting_id` values and generate ground-truth matches from `label_group`.


In [7]:
train_path = kagglehub.competition_download(
    "shopee-product-matching",
    path="train.csv",
    force_download=True
    )

100%|██████████| 2.48M/2.48M [00:01<00:00, 2.21MB/s]

Extracting zip of train.csv...


In [8]:
train_img_emb = np.load("/content/drive/MyDrive/shopee/artifacts/embeddings/efficientnet_b0_train.npz",
                    allow_pickle=True)
val_img_emb = np.load("/content/drive/MyDrive/shopee/artifacts/embeddings/efficientnet_b0_val.npz",
                  allow_pickle=True)

In [9]:
full_df = pd.read_csv(train_path)

In [10]:
full_df = full_df.set_index('posting_id')

train_df = full_df.loc[train_img_emb['posting_ids']].reset_index()
val_df = full_df.loc[val_img_emb['posting_ids']].reset_index()

full_df = full_df.reset_index()

In [11]:
target_matches = val_df.groupby("label_group")["posting_id"].agg(list)

val_df["target_matches"] = val_df["label_group"].map(target_matches)


## 3. Text Features — TF-IDF

Extract word-level unigram and bigram features. The vectorizer is fitted on the training set and applied to the validation set.


In [12]:
vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), max_features=27000)

train_txt_emb, val_txt_emb = get_tfidf(
    model=vectorizer,
    val_df=val_df,
    train_df=train_df
)

In [13]:
val_txt_emb

{'embeddings': <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 63042 stored elements and shape (6857, 27000)>,
 'posting_ids': array(['train_1806152124', 'train_3054371232', 'train_2961381387', ...,
        'train_1405089202', 'train_945815402', 'train_769054909'],
       dtype=object)}


## 4. Multimodal Retrieval

Retrieve candidates independently using image and text features with cosine distance.

- Gallery: train + validation
- Queries: validation


In [14]:
img_retrieval = image_retrieval(
    val_emb=val_img_emb,
    train_emb=train_img_emb,
    radius = 0.38
)

txt_retrieval = text_retrieval(
    val_emb=val_txt_emb,
    train_emb=train_txt_emb,
    radius = 0.38
)


## 5. Threshold Optimization & Fusion

Search for the best combination of image and text distance thresholds. Candidate matches are combined using union fusion and evaluated with Mean F1.

In [15]:
f1_scores = {}
for img_t in np.arange(0.13, 0.2, 0.01):
    img_matching_preds = threshold_filtering(
        neighbours=img_retrieval['indices'],
        distances=img_retrieval['distances'],
        gallery_ids=img_retrieval['gallery_ids'],
        query_ids=img_retrieval['query_ids'],
        threshold=img_t
    )
    for txt_t in np.arange(0.25, 0.325, 0.01):
      text_matching_preds = threshold_filtering(
          neighbours=txt_retrieval['indices'],
          distances=txt_retrieval['distances'],
          gallery_ids=txt_retrieval['gallery_ids'],
          query_ids=txt_retrieval['query_ids'],
          threshold=txt_t
      )
      preds = multimodal_fusion(
          image_matches=img_matching_preds,
          text_matches=text_matching_preds
      )
      f1_scores[(img_t, txt_t)] = (evaluate_predictions(val_df=val_df, preds=preds))

In [16]:
f1_scores = dict(sorted(f1_scores.items(), key=lambda item: item[1], reverse=True))


## 6. Results & Conclusions

**Best validation results:**

- Image threshold: 0.16
- Text threshold: 0.28
- Mean F1: 0.72832

Thresholds were optimized on a single validation split. Cross-validation is required to assess their stability and generalization.


In [17]:
print("Top 5 threshold combinations:\n")

for (img_t, txt_t), f1 in list(f1_scores.items())[:5]:
    print(f"Image: {img_t:.2f} | Text: {txt_t:.2f} | Mean F1: {f1:.5f}")

Top 5 threshold combinations:

Image: 0.16 | Text: 0.28 | Mean F1: 0.72832
Image: 0.16 | Text: 0.29 | Mean F1: 0.72798
Image: 0.16 | Text: 0.26 | Mean F1: 0.72792
Image: 0.16 | Text: 0.30 | Mean F1: 0.72790
Image: 0.16 | Text: 0.27 | Mean F1: 0.72779
